In [9]:
!pip install pulp

In [14]:

import pandas as pd
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Registrar el tiempo inicial
start_time = time.time()
Parametro_tiempo = 120
Inventario_inicio = []
Inventario_final = []
Estacion = []
estaciones_flujo = []
valores_flujo = []
limites_superiores = []
limites_inferiores = []
# Leer datos desde los archivos Excel
inventarios_df = pd.read_excel(r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\bikes_disponibles_2024-11-15_18-00.xlsx')  # Inventario inicial B_i,t
promedios_df = pd.read_excel(r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_20.xlsx')      # Promedios históricos b_i
asimetria_df = pd.read_excel(r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Asimetria_E20_variacion.xlsx')      # Asimetría histórica A_i
capacidades_df = pd.read_excel(r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20.xlsx')  # Capacidades M_i
costos_df = pd.read_excel(r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_20.xlsx')            # Costos CS_{i,j} y CE_{j,i}

print("datos leidos")
# Verificar y reemplazar valores NaN
def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

inventarios_df = fill_na(inventarios_df)
promedios_df = fill_na(promedios_df)
asimetria_df = fill_na(asimetria_df)
capacidades_df = fill_na(capacidades_df)
costos_df = fill_na(costos_df)

# Crear un mapeo de nombres de estaciones a índices
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
print('mapeo nombres')

# Parámetros del problema
I = len(unique_stations)  # Número de estaciones
T = 1                     # Consideramos un instante de tiempo t=1

# Leer valores de los DataFrames para los parámetros
M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).to_numpy()  # Capacidades de las estaciones
b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).to_numpy()     # Promedios históricos
MAX = asimetria_df.set_index('Estacion')['Max(0,ASI)'].reindex(unique_stations).to_numpy() # Asimetría histórica

# Inicializar matriz de costos
C = np.zeros((I, I))
for _, row in costos_df.iterrows():
    i = station_to_index[row['Estacion_Origen']]
    j = station_to_index[row['Estacion_Destino']]
    C[i, j] = row['Costo_Salida']  # Usar Costo_Salida como único costo por simplicidad

# Crear el problema de optimización
model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)
print('modelo definido')
# Definir variables de decisión
B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer")
      for i in range(I) for j in range(I) if i != j for t in range(T)}
Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary")
      for i in range(I) for j in range(I) if i != j for t in range(T)}

# Definir la función objetivo: minimizar costos basados en las distancias para lotes
model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j), "Funcion_Objetivo"
print('funcion obj definida')
# Restricciones

# 1. Restricción de Balance de Inventario en el tiempo t
for i in range(I):
    inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
    model += B[i, 0] == inventario_inicial, f"Balance_Inventario_Inicial_{i}_0"
    model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i), f"Balance_Inventario_{i}_1"

# 2. Restricción sobre la Capacidad Máxima de la Estación B_{i,t}
for i in range(I):
    model += B[i, 1] <= M[i], f"Capacidad_Maxima_{i}_1"

# 3. Restricción sobre las Bicicletas Enviadas FS_{i,j,t}
# Asegurar que el flujo total enviado desde una estación no excede su inventario inicial
for i in range(I):
    model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0], f"Total_Flow_Out_{i}_0"

# 4. Relación entre FS e Y: si hay flujo, entonces Y debe ser 1
for i in range(I):
    for j in range(I):
        if i != j:
            model += FS[i, j, 0] <= M[i] * Y[i, j, 0], f"Rel_FS_Y_{i}_{j}_0"

# 5. Restricción Superior del Inventario B_{i,t}
for i in range(I):
    ajuste_superior = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
    model += B[i, 1] <= ajuste_superior, f"Limite_Superior_{i}_1"
    limites_superiores.append(ajuste_superior)  # Guardar el valor calculado

# 6. Restricción Inferior del Inventario B_{i,t}
for i in range(I):
    ajuste_inferior = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
    model += B[i, 1] >= ajuste_inferior, f"Limite_Inferior_{i}_1"
    limites_inferiores.append(ajuste_inferior) 
limites_combinados = [[float(limites_superiores[i]), float(limites_inferiores[i])] for i in range(I)]
#print(model) #si quiero ver los limites inferior y superior
print('restricciones ingresadas')
# Resolver el problema utilizando el solver CBC (el solver predeterminado de Pulp)
status = model.solve(PULP_CBC_CMD(msg=True,timeLimit=Parametro_tiempo))
print('modelo resuelto')
# Calcular el costo total de la función objetivo
costo_total = model.objective.value()
print('calculo de costo total')
# Mostrar resultados
print("\nResultados:")
print("\nFunción Objetivo Completa:")
funcion_objetivo = "Minimizar: "
for i in range(I):
    for j in range(I):
        if i != j:
            funcion_objetivo += f"({C[i, j]} * Y[{unique_stations[i]}, {unique_stations[j]}]) + "
funcion_objetivo = funcion_objetivo.rstrip(" + ")
print(funcion_objetivo)

contador = 0  #Inicializar el contador

for i in range(I):
    for j in range(I):
        if i != j:
            flujo = FS[i, j, 0].varValue  # Obtener el valor de la variable
            print(f"Flujo Enviado de {unique_stations[i]} a {unique_stations[j]} en tiempo 0: {flujo}")
            if flujo > 0:  # Verificar si el valor es mayor que 0
                estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                valores_flujo.append(flujo) 
                contador += 1
  

print(f"El número total de flujos mayores a 0 es: {contador}")

print("\nRestricciones:")
print("1. Balance de Inventarios:")
print("   B[i, 0] = Inventario inicial de la estación i")
print("   B[i, 1] = B[i, 0] + Σ(FS[j, i, 0]) - Σ(FS[i, j, 0])")
print("2. Capacidad Máxima:")
print("   B[i, 1] <= Capacidad máxima de la estación i")
print("3. Flujos máximos por inventario inicial:")
print("   Σ(FS[i, j, 0]) <= B[i, 0]")
print("4. Relación flujo-binario:")
print("   FS[i, j, 0] <= M[i] * Y[i, j, 0]")
print("5. Límites superiores del inventario:")
print("   B[i, 1] <= (1+(b_[i]/M_[i])+(Max(0,ASI)/M[i]))*(M_[i]/2)")
print("6. Límites inferiores del inventario:")
print("   B[i, 1] >= (1-(b_[i]/M_[i])+(Max(0,ASI)/M[i]))*(M_[i]/2)")

print(f"\nCosto total de la función objetivo calculado por el modelo: {costo_total}")

# Mostrar inventarios iniciales,finales asimetrias y promedio....esto permite analizar el comportamiento historico de cada estación y porque se rebalancea de esa forma
print("\nInventarios Iniciales y Finales por Estación:")
for i in range(I):
    inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
    print(f"Estación {unique_stations[i]} - Asimetria:{MAX[i]},Promedio:{b[i]}, Inicial: {inventario_inicial}, Final: {B[i, 1].varValue}")

    Inventario_inicio.append(int(inventario_inicial)),
    Inventario_final.append(int(B[i, 1].varValue)),
    Estacion.append(unique_stations[i])

# Mostrar flujos entre estaciones
print("\nFlujos de Bicicletas entre Estaciones:")
for i in range(I):
    for j in range(I):
        if i != j and FS[i, j, 0].varValue>0:
            print(f"De {unique_stations[i]} a {unique_stations[j]}: {FS[i, j, 0].varValue}")
end_time = time.time()

# Calcular el tiempo transcurrido
elapsed_time = end_time - start_time

# Mostrar el resultado
print(f"El código se ejecutó en {elapsed_time:.4f} segundos.")

df = pd.DataFrame({
    "Fecha": [inventarios_df['Fecha_Hora'].tolist()[0]],
    "Costo_total": [costo_total],
    "Parametro_tiempo": [Parametro_tiempo],  # Haz que esto sea una lista
    "Tiempo_de_Ejecucion": [elapsed_time],
    "Estaciones" : [Estacion],
    "Inventario_inicial": [inventarios_df['Bikes disponibles'].tolist()],
    "Asimetria" : [asimetria_df['Asi'].tolist()],
    "Inventario_final": [Inventario_final],
    "Cantidad Total de Flujos": [contador],
    "Estaciones_Flujos": [estaciones_flujo],
    "Cantidad_Flujos" : [valores_flujo],
    "Limites" : [limites_combinados]
})

df
output_file = "Resultado_optimizacioncaso1analisis.xlsx"
hoja = "Sheet1"


print(f"Resultados guardados en: {output_file}")
# Convertir a DataFrame
df_nuevo = pd.DataFrame(df)

try:
    # Cargar el archivo existente
    book = load_workbook(output_file)
    with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
        # Seleccionar la hoja específica y encontrar la última fila
        startrow = writer.sheets[hoja].max_row if hoja in writer.sheets else 0

        # Escribir nuevos datos
        df_nuevo.to_excel(writer, sheet_name=hoja, index=False, header=not book.active.max_row, startrow=startrow)
except FileNotFoundError:
    # Si no existe el archivo, lo crea
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_nuevo.to_excel(writer, sheet_name=hoja, index=False)
    


datos leidos
mapeo nombres
modelo definido
funcion obj definida
restricciones ingresadas
modelo resuelto
calculo de costo total

Resultados:

Función Objetivo Completa:
Minimizar: (1.182110713123325 * Y[LC009 - Municipalidad De Las Condes, LC010 - Metro Manquehue - Norte]) + (2.125409689093058 * Y[LC009 - Municipalidad De Las Condes, LC011 - Metro Hernando De Magallanes]) + (1.499479283868792 * Y[LC009 - Municipalidad De Las Condes, LC019 - Metro Alcántara]) + (2.251714990283462 * Y[LC009 - Municipalidad De Las Condes, LC033 - Vitacura / San Patricio]) + (2.526835163536421 * Y[LC009 - Municipalidad De Las Condes, LC034 - Zurich]) + (0.7871934356365508 * Y[LC009 - Municipalidad De Las Condes, LC047 - Apoquindo / Luiz Zegers]) + (0.5062856617063619 * Y[LC009 - Municipalidad De Las Condes, LC066 - Cerro Colorado / Rosario Norte]) + (1.185489390966194 * Y[LC009 - Municipalidad De Las Condes, LC068 - Metro Manquehue - Sur]) + (2.472761636375546 * Y[LC009 - Municipalidad De Las Condes, LC097

In [4]:
#prueba modificacion
import os
import pandas as pd
import numpy as np
import time
from datetime import datetime
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
from openpyxl import load_workbook

# Ruta de las carpetas y archivos fijos
carpeta_inventarios = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todas'
archivo_asimetria = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx'
archivo_promedios = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx'
archivo_capacidades = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx'
archivo_costos = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx'

# Hojas de asimetría a iterar
hojas_asimetria = ['Asi-50','Asi-25','Asi', 'Asi+25', 'Asi+50']


# Inicializar archivo de salida
output_file = "Resultado_optimizaciontodasanalisisPRUEBA"
hoja = "Sheet1"

# Cargar datos estáticos que no cambian entre ejecuciones
promedios_df = pd.read_excel(archivo_promedios)
capacidades_df = pd.read_excel(archivo_capacidades)
costos_df = pd.read_excel(archivo_costos)

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1
Parametro_tiempo = 120

# Inicializar lista para recopilar resultados
resultados = []

# Recorrer todos los archivos de inventario
for nombre_archivo in os.listdir(carpeta_inventarios):
    if nombre_archivo.endswith(".xlsx"):
        ruta_inventario = os.path.join(carpeta_inventarios, nombre_archivo)
        inventarios_df = pd.read_excel(ruta_inventario)

        # Limpiar NaNs
        inventarios_df = inventarios_df.fillna(0)

        for hoja_asimetria in hojas_asimetria:
            asimetria_df = pd.read_excel(archivo_asimetria).fillna(0)

            # Iniciar tiempo de ejecución
            start_time = time.time()

            # Parámetros
            M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).to_numpy()
            b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).to_numpy()
            MAX = asimetria_df.set_index('Estacion')[hoja_asimetria].reindex(unique_stations).to_numpy()

            # Matriz de costos
            C = np.zeros((I, I))
            for _, row in costos_df.iterrows():
                i = station_to_index[row['Estacion_Origen']]
                j = station_to_index[row['Estacion_Destino']]
                C[i, j] = row['Costo_Salida']

            # Variables del modelo
            model = LpProblem(name="rebalanceo", sense=LpMinimize)
            B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
            FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
            Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

            # Función objetivo
            model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

            # Restricciones
            for i in range(I):
                inv_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
                model += B[i, 0] == inv_inicial
                model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)
                model += B[i, 1] <= M[i]
                model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
                model += B[i, 1] <= (1 + (b[i]/M[i]) + (MAX[i]/M[i])) * (M[i]/2)
                model += B[i, 1] >= (1 - (b[i]/M[i]) + (MAX[i]/M[i])) * (M[i]/2)
                for j in range(I):
                    if i != j:
                        model += FS[i, j, 0] <= M[i] * Y[i, j, 0]

            # Resolver
            model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
            costo_total = model.objective.value()
            elapsed_time = time.time() - start_time

            # Recopilar resultados
            fecha = pd.to_datetime(inventarios_df['Fecha_Hora'].iloc[0])
            estaciones_flujo = []
            valores_flujo = []
            contador = 0
            Inventario_final = []

            for i in range(I):
                Inventario_final.append(int(B[i, 1].varValue))
                for j in range(I):
                    if i != j and FS[i, j, 0].varValue > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(FS[i, j, 0].varValue)
                        contador += 1

            resultados.append({
                "Fecha": fecha,
                "Asimetria": hoja_asimetria,
                "Costo_total": costo_total,
                "Parametro_tiempo": Parametro_tiempo,
                "Tiempo_de_Ejecucion": elapsed_time,
                "Estaciones": unique_stations,
                "Inventario_inicial": inventarios_df['Bikes disponibles'].tolist(),
                "Inventario_final": Inventario_final,
                "Cantidad Total de Flujos": contador,
                "Estaciones_Flujos": estaciones_flujo,
                "Cantidad_Flujos": valores_flujo
            })

# Convertir resultados a DataFrame y guardar
df_resultado = pd.DataFrame(resultados)

try:
    book = load_workbook(output_file)
    with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
        startrow = writer.sheets[hoja].max_row if hoja in writer.sheets else 0
        df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=not book.active.max_row, startrow=startrow)
except FileNotFoundError:
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_resultado.to_excel(writer, sheet_name=hoja, index=False)

print(f"✅ Resultados guardados en {output_file}")


InvalidFileException: openpyxl does not support  file format, please check you can open it with Excel first. Supported formats are: .xlsx,.xlsm,.xltx,.xltm

In [ ]:
#prueba modificacion bebe
import os
import pandas as pd
import numpy as np
import time
from datetime import datetime
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
from openpyxl import load_workbook

# Ruta de las carpetas y archivos fijos
carpeta_inventarios = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todasbebe'
archivo_asimetria = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx'
archivo_promedios = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx'
archivo_capacidades = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx'
archivo_costos = r'C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx'

# Hojas de asimetría a iterar
hojas_asimetria = ['Asi-50','Asi-25','Asi', 'Asi+25', 'Asi+50']


# Inicializar archivo de salida
output_file = "Resultado_optimizaciontodasanalisisbebe"
hoja = "Sheet1"

# Cargar datos estáticos que no cambian entre ejecuciones
promedios_df = pd.read_excel(archivo_promedios)
capacidades_df = pd.read_excel(archivo_capacidades)
costos_df = pd.read_excel(archivo_costos)

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1
Parametro_tiempo = 120

# Inicializar lista para recopilar resultados
resultados = []

# Recorrer todos los archivos de inventario
for nombre_archivo in os.listdir(carpeta_inventarios):
    if nombre_archivo.endswith(".xlsx"):
        ruta_inventario = os.path.join(carpeta_inventarios, nombre_archivo)
        inventarios_df = pd.read_excel(ruta_inventario)

        # Limpiar NaNs
        inventarios_df = inventarios_df.fillna(0)

        for hoja_asimetria in hojas_asimetria:
            asimetria_df = pd.read_excel(archivo_asimetria).fillna(0)

            # Iniciar tiempo de ejecución
            start_time = time.time()

            # Parámetros
            M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).to_numpy()
            b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).to_numpy()
            MAX = asimetria_df.set_index('Estacion')[hoja_asimetria].reindex(unique_stations).to_numpy()

            # Matriz de costos
            C = np.zeros((I, I))
            for _, row in costos_df.iterrows():
                i = station_to_index[row['Estacion_Origen']]
                j = station_to_index[row['Estacion_Destino']]
                C[i, j] = row['Costo_Salida']

            # Variables del modelo
            model = LpProblem(name="rebalanceo", sense=LpMinimize)
            B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
            FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
            Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

            # Función objetivo
            model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

            # Restricciones
            for i in range(I):
                inv_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
                model += B[i, 0] == inv_inicial
                model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)
                model += B[i, 1] <= M[i]
                model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
                model += B[i, 1] <= (1 + (b[i]/M[i]) + (MAX[i]/M[i])) * (M[i]/2)
                model += B[i, 1] >= (1 - (b[i]/M[i]) + (MAX[i]/M[i])) * (M[i]/2)
                for j in range(I):
                    if i != j:
                        model += FS[i, j, 0] <= M[i] * Y[i, j, 0]

            # Resolver
            model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
            costo_total = model.objective.value()
            elapsed_time = time.time() - start_time

            # Recopilar resultados
            fecha = pd.to_datetime(inventarios_df['Fecha_Hora'].iloc[0])
            estaciones_flujo = []
            valores_flujo = []
            contador = 0
            Inventario_final = []

            for i in range(I):
                Inventario_final.append(int(B[i, 1].varValue))
                for j in range(I):
                    if i != j and FS[i, j, 0].varValue > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(FS[i, j, 0].varValue)
                        contador += 1

            resultados.append({
                "Fecha": fecha,
                "Asimetria": hoja_asimetria,
                "Costo_total": costo_total,
                "Parametro_tiempo": Parametro_tiempo,
                "Tiempo_de_Ejecucion": elapsed_time,
                "Estaciones": unique_stations,
                "Inventario_inicial": inventarios_df['Bikes disponibles'].tolist(),
                "Inventario_final": Inventario_final,
                "Cantidad Total de Flujos": contador,
                "Estaciones_Flujos": estaciones_flujo,
                "Cantidad_Flujos": valores_flujo
            })

# Convertir resultados a DataFrame y guardar
df_resultado = pd.DataFrame(resultados)

try:
    book = load_workbook(output_file)
    with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
        startrow = writer.sheets[hoja].max_row if hoja in writer.sheets else 0
        df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=not book.active.max_row
                              , startrow=startrow)
except FileNotFoundError:
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_resultado.to_excel(writer, sheet_name=hoja, index=False)

print(f"✅ Resultados guardados en {output_file}")


In [15]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Parámetros globales
directorio_inventarios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todasbebe"
archivo_asimetria = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx"
archivo_promedios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx"
archivo_capacidades = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx"
archivo_costos = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx"

output_file = "Resultado_optimizaciontodasanalisisbebe.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])
columnas_asimetria = [col for col in asimetria_df_completo.columns if col != "Estacion"]

for archivo_inv in archivos_inventario:
    inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))
    for columna_asim in columnas_asimetria:
        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con asimetría {columna_asim}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).to_numpy()
        MAX = asimetria_df_completo.set_index('Estacion')[columna_asim].reindex(unique_stations).to_numpy()

        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]
            ajuste_sup = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Asimetria": [columna_asim],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[(float(limites_superiores[i]), float(limites_inferiores[i])) for i in range(I)]]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)

        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")


Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría 0
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Asi+50
Ejecutando: bikes_disponibles_2024-11-08_18-00.xlsx con asimetría Max(0,ASI)
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría 0
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi-50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi-25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi+25
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetría Asi+50
Ejecutando: bikes_disponibles_2024-11-09_18-00.xlsx con asimetr

In [ ]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook

# Parámetros globales
directorio_inventarios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todasbebe"
archivo_asimetria = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx"
archivo_promedios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx"
archivo_capacidades = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx"
archivo_costos = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx"

output_file = "Resultado_optimizaciontodasanalisisbebe.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])

for archivo_inv in archivos_inventario:
    inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))
    
    for fila_idx in range(asimetria_df_completo.shape[0]):  # cada fila representa una variación
        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con fila de asimetría {fila_idx}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).fillna(0).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).fillna(1).to_numpy()

        # Extraer vector MAX desde la fila correspondiente
        fila_asimetria = asimetria_df_completo.iloc[fila_idx]
        estacion = fila_asimetria['Estacion']
        valor_asimetria = fila_asimetria.drop(labels='Estacion')  # los valores de la fila sin la columna "Estacion"
        MAX = valor_asimetria.reindex(unique_stations).fillna(0).to_numpy()

        # Crear matriz de costos
        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        # MODELO
        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]

            ajuste_sup = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[(float(limites_superiores[i]), float(limites_inferiores[i])) for i in range(I)]],
            "Variacion asi": [fila_idx]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)

        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")


In [ ]:
import numpy as np
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, PULP_CBC_CMD
import pandas as pd
import time
import os
from openpyxl import load_workbook
from datetime import datetime

# Parámetros globales
directorio_inventarios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\invdiarios_todasbebe"
archivo_asimetria = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\asimetria_bikes_todascom_+20%.xlsx"
archivo_promedios = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\promedio_bikes_por_estacion_todas.xlsx"
archivo_capacidades = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\Capacidades_elegidas20todascom.xlsx"
archivo_costos = r"C:\Users\ACER\Desktop\Tesis\Tesis Pedro Palominos\Código python\Modelo_info_historica\Modelo más actualizado\analisis sensibilidad\analisis aumentar\distancias_estaciones_todascom.xlsx"

output_file = "Resultado_optimizaciontodasanalisisbebe.xlsx"
hoja = "Sheet1"
Parametro_tiempo = 120

def fill_na(dataframe, fill_value=0):
    return dataframe.fillna(fill_value)

# Leer archivos fijos
promedios_df = fill_na(pd.read_excel(archivo_promedios))
capacidades_df = fill_na(pd.read_excel(archivo_capacidades))
costos_df = fill_na(pd.read_excel(archivo_costos))
asimetria_df_completo = fill_na(pd.read_excel(archivo_asimetria))

# Mapeo de estaciones
unique_stations = sorted(list(set(list(costos_df['Estacion_Origen'].unique()) + list(costos_df['Estacion_Destino'].unique()))))
station_to_index = {name: idx for idx, name in enumerate(unique_stations)}
I = len(unique_stations)
T = 1

# Preparar iteración
archivos_inventario = sorted([f for f in os.listdir(directorio_inventarios) if f.endswith(".xlsx")])
columnas_asimetria = ['Asi-50', 'Asi-25', 'Asi', 'Asi+25', 'Asi+50']

for archivo_inv in archivos_inventario:
    inventarios_df = fill_na(pd.read_excel(os.path.join(directorio_inventarios, archivo_inv)))

    for columna_asim in columnas_asimetria:
        start_time = time.time()
        print(f"Ejecutando: {archivo_inv} con asimetría {columna_asim}")

        Inventario_inicio, Inventario_final, Estacion = [], [], []
        estaciones_flujo, valores_flujo = [], []
        limites_superiores, limites_inferiores = [], []

        b = promedios_df.set_index('Estacion')['Promedio Bikes'].reindex(unique_stations).fillna(0).to_numpy()
        M = capacidades_df.set_index('Estacion')['Capacidad'].reindex(unique_stations).fillna(1).to_numpy()

        # ✅ APLICA max(0, ASI) para la columna correspondiente
        asi_values = asimetria_df_completo.set_index('Estacion')[columna_asim].reindex(unique_stations).fillna(0).to_numpy()
        MAX = np.maximum(0, asi_values)

        # MATRIZ DE COSTOS
        C = np.zeros((I, I))
        for _, row in costos_df.iterrows():
            i = station_to_index[row['Estacion_Origen']]
            j = station_to_index[row['Estacion_Destino']]
            C[i, j] = row['Costo_Salida']

        # CREACIÓN DEL MODELO
        model = LpProblem(name="problema-rebalanceo-bicicletas", sense=LpMinimize)

        B = {(i, t): LpVariable(f"B_{i}_{t}", lowBound=0, cat="Integer") for i in range(I) for t in range(T+1)}
        FS = {(i, j, t): LpVariable(f"FS_{i}_{j}_{t}", lowBound=0, cat="Integer") for i in range(I) for j in range(I) if i != j for t in range(T)}
        Y = {(i, j, t): LpVariable(f"Y_{i}_{j}_{t}", cat="Binary") for i in range(I) for j in range(I) if i != j for t in range(T)}

        model += lpSum(C[i, j] * Y[i, j, 0] for i in range(I) for j in range(I) if i != j)

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            model += B[i, 0] == inventario_inicial
            model += B[i, 1] == B[i, 0] + lpSum(FS[j, i, 0] for j in range(I) if j != i) - lpSum(FS[i, j, 0] for j in range(I) if j != i)

        for i in range(I):
            model += B[i, 1] <= M[i]
            model += lpSum(FS[i, j, 0] for j in range(I) if j != i) <= B[i, 0]
            for j in range(I):
                if i != j:
                    model += FS[i, j, 0] <= M[i] * Y[i, j, 0]
            ajuste_sup = (1 + (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            ajuste_inf = (1 - (b[i] / M[i]) + (MAX[i] / M[i])) * (M[i] / 2)
            model += B[i, 1] <= ajuste_sup
            model += B[i, 1] >= ajuste_inf
            limites_superiores.append(ajuste_sup)
            limites_inferiores.append(ajuste_inf)

        status = model.solve(PULP_CBC_CMD(msg=False, timeLimit=Parametro_tiempo))
        costo_total = model.objective.value()
        contador = 0
        for i in range(I):
            for j in range(I):
                if i != j:
                    flujo = FS[i, j, 0].varValue
                    if flujo > 0:
                        estaciones_flujo.append(f"{unique_stations[i]} -> {unique_stations[j]}")
                        valores_flujo.append(flujo)
                        contador += 1

        for i in range(I):
            inventario_inicial = inventarios_df.loc[inventarios_df['Estacion'] == unique_stations[i], 'Bikes disponibles'].values[0]
            Inventario_inicio.append(int(inventario_inicial))
            Inventario_final.append(int(B[i, 1].varValue))
            Estacion.append(unique_stations[i])

        elapsed_time = time.time() - start_time
        fecha_valor = inventarios_df['Fecha_Hora'].tolist()[0] if 'Fecha_Hora' in inventarios_df.columns else archivo_inv

        df_resultado = pd.DataFrame({
            "Fecha": [fecha_valor],
            "Costo_total": [costo_total],
            "Parametro_tiempo": [Parametro_tiempo],
            "Tiempo_de_Ejecucion": [elapsed_time],
            "Estaciones": [Estacion],
            "Inventario_inicial": [Inventario_inicio],
            "Asimetria": [columna_asim],
            "Inventario_final": [Inventario_final],
            "Cantidad Total de Flujos": [contador],
            "Estaciones_Flujos": [estaciones_flujo],
            "Cantidad_Flujos": [valores_flujo],
            "Limites": [[(float(limites_superiores[i]), float(limites_inferiores[i])) for i in range(I)]]
        })

        try:
            if os.path.exists(output_file):
                book = load_workbook(output_file)
                if hoja in book.sheetnames:
                    startrow = book[hoja].max_row
                else:
                    startrow = 0
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False, header=startrow == 0, startrow=startrow)
            else:
                with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
                    df_resultado.to_excel(writer, sheet_name=hoja, index=False)

        except Exception as e:
            print(f"Error al guardar resultados: {e}")

print("¡Todos los modelos han sido ejecutados y guardados correctamente!")
